In [1]:
import warnings
warnings.filterwarnings('ignore')

In [2]:
import torch
import torch.nn as nn

In [3]:
data = [
    ("I love machine", "learning"),
    ("I love deep", "learning"),
    ("I like machine", "learning"),
    ("I like deep", "learning"),
    ("machine learning is", "powerful"),
    ("deep learning is", "powerful"),
    ("Python is very", "useful"),
    ("PyTorch is very", "useful"),
    ("AI is curcial for out", "task"),
    ("GRU is very", "powerful"),
]

In [4]:
def build_vocab(sentences):
    vocab = {
        "<pad>": 0,
        "<unk>": 1
    }

    for sentence in sentences:

        for word in sentence.split():
            if word not in vocab:
                vocab[word] = len(vocab)
    return vocab

input_vocab = build_vocab(x[0] for x in data)
target_vocab = build_vocab(x[1] for x in data)

In [5]:
print(input_vocab)
print(target_vocab)

{'<pad>': 0, '<unk>': 1, 'I': 2, 'love': 3, 'machine': 4, 'deep': 5, 'like': 6, 'learning': 7, 'is': 8, 'Python': 9, 'very': 10, 'PyTorch': 11, 'AI': 12, 'curcial': 13, 'for': 14, 'out': 15, 'GRU': 16}
{'<pad>': 0, '<unk>': 1, 'learning': 2, 'powerful': 3, 'useful': 4, 'task': 5}


In [6]:
def text_to_number(sentence, vocab):
    return torch.tensor(
        [vocab.get(word, vocab['<unk>']) for word in sentence.split()],
        dtype=torch.long
    )

input = [text_to_number(sentence, input_vocab) for sentence, _ in data]
output = [text_to_number(sentence, target_vocab) for _, sentence in data]

In [7]:
print(input), print(output)

[tensor([2, 3, 4]), tensor([2, 3, 5]), tensor([2, 6, 4]), tensor([2, 6, 5]), tensor([4, 7, 8]), tensor([5, 7, 8]), tensor([ 9,  8, 10]), tensor([11,  8, 10]), tensor([12,  8, 13, 14, 15]), tensor([16,  8, 10])]
[tensor([2]), tensor([2]), tensor([2]), tensor([2]), tensor([3]), tensor([3]), tensor([4]), tensor([4]), tensor([5]), tensor([3])]


(None, None)

In [8]:
from torch.nn.utils.rnn import pad_sequence

input_pad = pad_sequence(
    input,
    batch_first=True,
    padding_value=input_vocab["<pad>"]
)

print(input_pad)

tensor([[ 2,  3,  4,  0,  0],
        [ 2,  3,  5,  0,  0],
        [ 2,  6,  4,  0,  0],
        [ 2,  6,  5,  0,  0],
        [ 4,  7,  8,  0,  0],
        [ 5,  7,  8,  0,  0],
        [ 9,  8, 10,  0,  0],
        [11,  8, 10,  0,  0],
        [12,  8, 13, 14, 15],
        [16,  8, 10,  0,  0]])


In [9]:
vocab_size = len(input_vocab)
embedding_dim = 10

embedding = nn.Embedding(
    num_embeddings=vocab_size,
    embedding_dim=embedding_dim
)

In [10]:
x = torch.tensor([2, 3, 4])
embedded = embedding(x)
print(embedded)

tensor([[ 0.4517, -1.5359,  0.3776, -0.0313,  0.5877, -1.1051,  0.7559, -1.3320,
          0.9372, -2.0137],
        [ 2.2804,  0.3584,  0.8510,  1.0539,  1.1267, -0.2911,  0.1920,  0.0997,
         -1.4008,  0.6275],
        [ 1.3874,  0.5232, -0.0080,  0.9566,  1.4867, -0.3756,  0.4880, -0.2023,
          0.3512, -0.1534]], grad_fn=<EmbeddingBackward0>)


In [17]:
class GRUClassifier(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, output_size):
        super().__init__()

        self.embedding = nn.Embedding(
            num_embeddings=vocab_size,
            embedding_dim=embedding_dim,
        )

        self.gru = nn.GRU(
            input_size=embedding_dim,
            hidden_size=hidden_dim
        )

        self.linear = nn.Linear(hidden_dim, output_size)

    def forward(self, sentence):
        embedded = self.embedding(sentence)
        hidden, output = self.gru(embedded)
        return nn.Linear(output)

In [18]:
learning_rate = 0.01
epochs = 10

In [19]:
model = GRUClassifier(vocab_size, 32, 128, len(output))
optim = torch.optim.Adam(model.parameters(), lr=learning_rate)
loss_fn = torch.nn.CrossEntropyLoss()